# Chapter 4: Lines and Cross-Ratios

**Source span.** Perspectives on Projective Geometry, Chapter 4, Sections 4.1-4.4, printed pp. 67-78 / PDF pp. 89-100.

**Chapter goal.** Learn how a projective line can carry coordinates without making those coordinates intrinsic, and identify the cross-ratio as the first numerical quantity in the chapter that survives projective reparametrization.

The source span moves through a tight chain of ideas: a line through two projective points is a two-dimensional linear span, the parameter pair on that span is itself a point of `RP^1`, a change of reference points acts by a `2 x 2` projective matrix, and the determinant formula for the cross-ratio is unaffected by scaling or by such a matrix. This notebook rebuilds that chain computationally. The figures are not page reproductions; they are new diagrams made from the coordinate rules so that each invariant can be tested.

**Visual route.**

1. Put coordinates on a line with representatives `lambda p + mu q`, then watch the affine charts `t = lambda/mu` and `u = mu/lambda` trade the finite coordinate with the point at infinity.
2. Apply a `2 x 2` projective matrix to `RP^1` and compare the before-and-after coordinate positions of four marked points.
3. Compute the cross-ratio from `2 x 2` determinants, then check the same value after scaling representatives, after a Mobius reparametrization, and from the `RP^2` determinant formula.
4. Use the condition `(a,b;c,d) = -1` to build harmonic quadruples, including the case where the fourth point is the infinite point.
5. Track the six cross-ratio values obtained by permuting the four entries, because the chapter's algebra depends on the order of the points.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
search_roots = [START, *START.parents, START / "Perspectives-on-Projective-Geometry"]
search_roots.extend(child for child in START.glob("*") if child.is_dir())

BOOK_ROOT = None
for candidate in search_roots:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate.resolve()
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives on Projective Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-04-lines-and-cross-ratios"
for child in ("figures", "html", "tables", "checks"):
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = BOOK_ROOT / "part-01-projective-geometry" / "chapter-04-lines-and-cross-ratios" / "04-lines-and-cross-ratios.ipynb"
BOOK_ROOT, ARTIFACT_ROOT


## Computational Translation Guide

| Chapter idea | Computational representation | What must not change |
| --- | --- | --- |
| Points on the line through `[p]` and `[q]` | Nonzero pairs `(lambda, mu)` mapped to `[lambda p + mu q]` | Multiplying both parameters by the same nonzero scalar |
| The real projective line `RP^1` | Nonzero vectors `[x:y]`, with finite chart `t = x/y` when `y != 0` | The projective point represented by the ray, not the chart value |
| Chart transition | On the overlap, `u = y/x = 1/t` | Finite points agree in both charts; infinity is chart-dependent |
| Projective reparametrization | Matrix action `[x:y] -> M[x:y]`, finite formula `t -> (a t + b)/(c t + d)` | The cross-ratio of four points |
| Cross-ratio | Determinant quotient `[a,c][b,d]/([a,d][b,c])` | Representative scaling and invertible `2 x 2` matrices |
| Harmonic quadruple | Ordered quadruple with cross-ratio `-1` | The fourth point may be finite or infinite |

**Library choices.** Static `matplotlib` diagrams are used for one-dimensional coordinate geometry because labels, rays, and finite/infinite chart positions need durable inspection. `plotly` is used for the Mobius/infinity lab because zooming near a pole makes the projective compactification visible. `sympy` supplies exact determinant and permutation identities. `networkx` renders the small six-value permutation orbit as a proof-state graph rather than as decorative art.


In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import artifact_path, assert_artifacts, book_relative, display_artifact, image_stats, save_json

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

TOL = 1e-10


def rp1(value):
    if value is math.inf or (isinstance(value, float) and math.isinf(value)):
        return np.array([1.0, 0.0])
    return np.array([float(value), 1.0])


def affine_chart(vector, tol=TOL):
    vector = np.asarray(vector, dtype=float)
    if abs(vector[1]) < tol:
        return math.inf
    return float(vector[0] / vector[1])


def det2(a, b):
    return float(np.linalg.det(np.column_stack([a, b])))


def det3(a, b, c):
    return float(np.linalg.det(np.column_stack([a, b, c])))


def cross_ratio_hom(a, b, c, d):
    return (det2(a, c) * det2(b, d)) / (det2(a, d) * det2(b, c))


def cross_ratio_affine(a, b, c, d):
    return cross_ratio_hom(rp1(a), rp1(b), rp1(c), rp1(d))


def mobius_apply(matrix, value):
    return affine_chart(np.asarray(matrix, dtype=float) @ rp1(value))


def harmonic_fourth(a, b, c, tol=TOL):
    denom = a + b - 2 * c
    if abs(denom) < tol:
        return math.inf
    return ((a - c) * b + (b - c) * a) / denom


def plane_point_on_x_axis(value):
    if value is math.inf or (isinstance(value, float) and math.isinf(value)):
        return np.array([1.0, 0.0, 0.0])
    return np.array([float(value), 0.0, 1.0])


def join3(p, q):
    return np.cross(np.asarray(p, dtype=float), np.asarray(q, dtype=float))


def meet3(l, m):
    return np.cross(np.asarray(l, dtype=float), np.asarray(m, dtype=float))


def cross_ratio_rp2(auxiliary, a, b, c, d):
    return (det3(auxiliary, a, c) * det3(auxiliary, b, d)) / (det3(auxiliary, a, d) * det3(auxiliary, b, c))


def save_figure(fig, filename):
    path = artifact_path(ARTIFACT_ROOT, "figures", filename)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def relative_stats(path):
    stats = image_stats(path)
    stats["path"] = book_relative(path)
    return stats


## Coordinates on One Projective Line

Choose two distinct projective points `[p]` and `[q]` in a plane. Every point on their line has a representative `lambda p + mu q`, and the pair `[lambda:mu]` is a point of `RP^1`. The point is unchanged if both parameters are scaled by the same nonzero number, so a single affine parameter `t = lambda/mu` only describes the chart where `mu != 0`. The missing chart point is `[1:0]`, the point at infinity for that coordinate.

The first artifact shows this as a pair of views. On the left, rays in `R^2` represent projective points `[x:y]` and the line `y = 1` is the finite chart. On the right, the same projective points are placed in the two reciprocal charts `t = x/y` and `u = y/x`; no finite chart owns the point at infinity.


In [ ]:
chart_values = [-2.0, -0.5, 0.8, 2.0, math.inf]
chart_colors = ["#4263eb", "#0b7285", "#2b8a3e", "#f08c00", "#c92a2a"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"width_ratios": [1.05, 1]})
ax = axes[0]
ax.axhline(0, color="#888", lw=0.8)
ax.axvline(0, color="#888", lw=0.8)
ax.plot([-2.6, 2.6], [1, 1], color="#222", lw=1.5, label="finite chart y=1")
ax.text(2.52, 1.08, "y=1", ha="right")
for value, color in zip(chart_values, chart_colors):
    if math.isinf(value):
        ax.plot([0, 2.7], [0, 0], color=color, lw=2.2)
        ax.scatter([2.35], [0], s=46, color=color, zorder=4)
        ax.text(2.18, -0.28, "[1:0]\ninfinity", color=color, ha="center", va="top")
        continue
    end = np.array([value, 1.0])
    ray = end / np.linalg.norm(end) * 2.6
    ax.plot([0, ray[0]], [0, ray[1]], color=color, lw=1.8)
    ax.scatter([value], [1], s=42, color=color, zorder=5)
    ax.text(value, 1.14, f"t={value:g}", color=color, ha="center")
ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-0.45, 2.15)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Projective points as rays")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.axhline(1, color="#333", lw=1.2)
ax.axhline(0, color="#333", lw=1.2)
ax.text(-3.0, 1.12, "finite chart t=x/y", va="bottom")
ax.text(-3.0, 0.12, "reciprocal chart u=y/x", va="bottom")
for value, color in zip(chart_values, chart_colors):
    if math.isinf(value):
        ax.scatter([2.88], [1], color=color, s=46, marker=">", zorder=5)
        ax.text(2.7, 1.2, "t goes to infinity", color=color, ha="right")
        ax.scatter([0], [0], color=color, s=46, zorder=5)
        ax.plot([0, 2.88], [0, 1], color=color, alpha=0.25, lw=1)
        ax.text(0, -0.18, "u=0", color=color, ha="center", va="top")
        continue
    u = 1 / value if value != 0 else math.inf
    ax.scatter([value], [1], color=color, s=46, zorder=5)
    ax.text(value, 1.2, f"{value:g}", color=color, ha="center")
    if math.isfinite(u) and abs(u) <= 3:
        ax.scatter([u], [0], color=color, s=46, zorder=5)
        ax.plot([value, u], [1, 0], color=color, alpha=0.25, lw=1)
        ax.text(u, -0.18, f"{u:g}", color=color, ha="center", va="top")
    else:
        side = 2.88 if u > 0 else -2.88
        ax.scatter([side], [0], color=color, s=46, marker=">" if u > 0 else "<", zorder=5)
        ax.plot([value, side], [1, 0], color=color, alpha=0.25, lw=1)
        ax.text(side, -0.18, "u large", color=color, ha="center", va="top")
ax.set_xlim(-3.15, 3.15)
ax.set_ylim(-0.45, 1.55)
ax.set_yticks([])
ax.set_title("The overlap uses u=1/t")
ax.spines[["left", "right", "top", "bottom"]].set_visible(False)
chart_transition_path = save_figure(fig, "projective-line-rp1-chart-transition.png")

overlap_values = np.array([-2.0, -0.5, 0.8, 2.0])
chart_transition_error = float(np.max(np.abs(overlap_values * (1 / overlap_values) - 1)))
book_relative(chart_transition_path), chart_transition_error


## Projective Reparametrization on `RP^1`

A `2 x 2` invertible matrix acts on homogeneous coordinates. In the finite chart this becomes the familiar Mobius formula

```text
t -> (a t + b) / (c t + d).
```

The formula can send a finite point to infinity when `c t + d = 0`, and it can bring the old point at infinity back to a finite value when `c != 0`. Because the matrix acts linearly on homogeneous representatives, determinants all acquire the same factor, and that common factor cancels in the cross-ratio.


In [ ]:
M = np.array([[1.10, 0.35], [-0.18, 1.00]])
tracked = np.array([-1.4, -0.25, 0.75, 1.8])
tracked_images = np.array([mobius_apply(M, float(t)) for t in tracked])
cr_before = cross_ratio_hom(*(rp1(float(t)) for t in tracked))
cr_after = cross_ratio_hom(*(M @ rp1(float(t)) for t in tracked))
mobius_error = float(abs(cr_before - cr_after))

fig, ax = plt.subplots(figsize=(10.5, 3.6))
ax.axhline(1, color="#222", lw=1.2)
ax.axhline(0, color="#222", lw=1.2)
ax.text(-3.4, 1.11, "input coordinate t", ha="left")
ax.text(-3.4, 0.11, "image coordinate t'", ha="left")
for idx, (t, image) in enumerate(zip(tracked, tracked_images), start=1):
    color = chart_colors[(idx - 1) % len(chart_colors)]
    ax.scatter([t], [1], color=color, s=54, zorder=5)
    ax.scatter([image], [0], color=color, s=54, zorder=5)
    ax.annotate("", xy=(image, 0.06), xytext=(t, 0.94), arrowprops=dict(arrowstyle="->", lw=1.2, color=color, alpha=0.65))
    ax.text(t, 1.23, f"p{idx}\n{t:.2f}", color=color, ha="center", va="bottom")
    ax.text(image, -0.18, f"p{idx}'\n{image:.2f}", color=color, ha="center", va="top")
pole = -M[1, 1] / M[1, 0]
ax.axvline(pole, color="#c92a2a", lw=1.2, ls="--", alpha=0.7)
ax.text(pole, 0.48, "denominator zero\n(point maps to infinity)", color="#c92a2a", ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.85))
ax.set_xlim(-3.55, 5.9)
ax.set_ylim(-0.55, 1.55)
ax.set_yticks([])
ax.set_xlabel("affine coordinate value")
ax.set_title(f"Cross-ratio before = {cr_before:.6f}; after = {cr_after:.6f}; residual = {mobius_error:.1e}")
ax.spines[["left", "right", "top"]].set_visible(False)
mobius_path = save_figure(fig, "mobius-reparametrization-cross-ratio.png")
book_relative(mobius_path), mobius_error


In [ ]:
def mobius_curve_trace(matrix, label, color, visible):
    matrix = np.asarray(matrix, dtype=float)
    pole = -matrix[1, 1] / matrix[1, 0] if abs(matrix[1, 0]) > TOL else None
    xs = np.linspace(-4.0, 4.0, 800)
    ys = []
    for x in xs:
        den = matrix[1, 0] * x + matrix[1, 1]
        if abs(den) < 0.035:
            ys.append(np.nan)
        else:
            ys.append((matrix[0, 0] * x + matrix[0, 1]) / den)
    traces = [go.Scatter(x=xs, y=ys, mode="lines", name=f"{label}: t'", line=dict(color=color, width=3), visible=visible)]
    images = [mobius_apply(matrix, float(t)) for t in tracked]
    traces.append(
        go.Scatter(
            x=tracked,
            y=images,
            mode="markers+text",
            text=[f"p{i}" for i in range(1, len(tracked) + 1)],
            textposition="top center",
            marker=dict(color=color, size=9, line=dict(color="white", width=1)),
            name=f"{label}: four points",
            visible=visible,
        )
    )
    if pole is not None and -4 <= pole <= 4:
        traces.append(go.Scatter(x=[pole, pole], y=[-8, 8], mode="lines", name=f"{label}: pole", line=dict(color=color, dash="dash", width=2), visible=visible))
    else:
        traces.append(go.Scatter(x=[], y=[], mode="lines", name=f"{label}: no visible pole", visible=visible))
    return traces


lab_matrices = [
    ("distant pole", np.array([[1.10, 0.35], [-0.18, 1.00]]), "#4263eb"),
    ("inversion", np.array([[0.00, 1.00], [1.00, 0.00]]), "#0b7285"),
    ("near pole", np.array([[1.00, -0.60], [0.45, 1.00]]), "#c92a2a"),
]
fig = go.Figure()
for i, (label, matrix, color) in enumerate(lab_matrices):
    for trace in mobius_curve_trace(matrix, label, color, visible=(i == 0)):
        fig.add_trace(trace)
buttons = []
for i, (label, matrix, color) in enumerate(lab_matrices):
    visible = [False] * (3 * len(lab_matrices))
    for j in range(3):
        visible[3 * i + j] = True
    cr_value = cross_ratio_hom(*(matrix @ rp1(float(t)) for t in tracked))
    buttons.append(dict(label=label, method="update", args=[{"visible": visible}, {"title": f"Mobius action on RP^1: {label}; cross-ratio = {cr_value:.6f}"}]))
fig.update_layout(
    title=f"Mobius action on RP^1: distant pole; cross-ratio = {cr_after:.6f}",
    xaxis_title="input affine coordinate t",
    yaxis_title="image coordinate t'",
    yaxis_range=[-8, 8],
    xaxis_range=[-4, 4],
    template="plotly_white",
    updatemenus=[dict(buttons=buttons, x=0.01, y=1.16, xanchor="left", yanchor="top")],
    annotations=[dict(text="Use the buttons, then zoom near a dashed pole to see a finite point pass through infinity.", x=0.5, y=-0.18, xref="paper", yref="paper", showarrow=False)],
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
html_path = artifact_path(ARTIFACT_ROOT, "html", "rp1-mobius-infinity-lab.html")
fig.write_html(html_path, include_plotlyjs=True, full_html=True)
book_relative(html_path)


## Cross-Ratio as a Determinant Invariant

For vectors `a,b,c,d` in `R^2`, the cross-ratio used here is

```text
(a,b;c,d) = [a,c][b,d] / ([a,d][b,c]),
```

where `[u,v]` is the determinant with columns `u` and `v`. This formula has two practical advantages over a distance ratio: it ignores representative scaling, and it still handles the infinite point. The next code cell checks the identities numerically and also checks the `RP^2` determinant version from the chapter: four collinear points in the plane may be measured using a fifth point off the line.


In [ ]:
scale_factors = [2.0, -3.0, 0.5, 4.0]
scaled_points = [factor * rp1(float(t)) for factor, t in zip(scale_factors, tracked)]
scaled_error = float(abs(cr_before - cross_ratio_hom(*scaled_points)))

aux = np.array([0.25, 1.15, 1.0])
plane_points = [plane_point_on_x_axis(float(t)) for t in tracked]
rp2_cr = cross_ratio_rp2(aux, *plane_points)
rp2_error = float(abs(cr_before - rp2_cr))

projection_center = np.array([0.35, 1.75, 1.0])
target_line = np.array([0.0, 1.0, -1.0])
projected_points = []
for point in plane_points:
    ray = join3(projection_center, point)
    image = meet3(ray, target_line)
    projected_points.append(image)
projection_aux = np.array([0.0, 0.0, 1.0])
projected_cr = cross_ratio_rp2(projection_aux, *projected_points)
projection_error = float(abs(cr_before - projected_cr))

invariance_rows = [
    {"check": "original finite chart", "cross_ratio": cr_before, "expected": cr_before, "residual": 0.0, "interpretation": "Four finite chart values measured by 2 x 2 determinants."},
    {"check": "scaled representatives", "cross_ratio": cross_ratio_hom(*scaled_points), "expected": cr_before, "residual": scaled_error, "interpretation": "Scaling each homogeneous representative cancels from the determinant quotient."},
    {"check": "Mobius image", "cross_ratio": cr_after, "expected": cr_before, "residual": mobius_error, "interpretation": "A 2 x 2 projective matrix changes coordinates but preserves the cross-ratio."},
    {"check": "RP2 determinant formula", "cross_ratio": rp2_cr, "expected": cr_before, "residual": rp2_error, "interpretation": "The same four points on a plane line measured with an auxiliary point off the line."},
    {"check": "perspective projection", "cross_ratio": projected_cr, "expected": cr_before, "residual": projection_error, "interpretation": "Projection from a viewpoint to another line induces a projective map."},
]
table_path = artifact_path(ARTIFACT_ROOT, "tables", "cross-ratio-invariance-table.csv")
pd.DataFrame(invariance_rows)[["check", "cross_ratio", "expected", "residual", "interpretation"]].to_csv(table_path, index=False)
pd.DataFrame(invariance_rows)


## Harmonic Quadruples and Crossing Infinity

A harmonic quadruple is an ordered quadruple with cross-ratio `-1`. If `a`, `b`, and `c` are finite and distinct, solving `(a,b;c,d) = -1` gives a fourth point. For `a=-1` and `b=1`, the formula simplifies to `d = 1/c`; as `c` passes through `0`, the fourth point crosses the infinite point of `RP^1` rather than disappearing.

This is the best place to see why a projective line is not just the ordinary real line with a loose endpoint added. The endpoint is part of the same one-dimensional object, and the determinant formula computes through it.


In [ ]:
a_h, b_h = -1.0, 1.0
c_h = 0.4
d_h = harmonic_fourth(a_h, b_h, c_h)
harmonic_cr = cross_ratio_affine(a_h, b_h, c_h, d_h)
infinity_harmonic_cr = cross_ratio_hom(rp1(a_h), rp1(b_h), rp1(0.0), rp1(math.inf))

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2), gridspec_kw={"width_ratios": [1, 1.2]})
ax = axes[0]
ax.axhline(0, color="#222", lw=1.2)
finite_marks = [(a_h, "a=-1", "#4263eb"), (b_h, "b=1", "#0b7285"), (c_h, "c=0.4", "#2b8a3e"), (d_h, f"d={d_h:.2f}", "#c92a2a")]
for x, label, color in finite_marks:
    ax.scatter([x], [0], color=color, s=70, zorder=4)
    ax.text(x, 0.13, label, color=color, ha="center", va="bottom")
ax.annotate("same equation with c=0 sends d to [1:0]", xy=(2.8, 0), xytext=(1.25, -0.42), arrowprops=dict(arrowstyle="->", color="#c92a2a"), color="#c92a2a")
ax.scatter([3.15], [0], marker=">", color="#c92a2a", s=70)
ax.text(3.0, 0.15, "infinity", color="#c92a2a", ha="right")
ax.set_xlim(-1.7, 3.35)
ax.set_ylim(-0.65, 0.55)
ax.set_yticks([])
ax.set_title(f"Finite harmonic example: cross-ratio = {harmonic_cr:.1f}")
ax.spines[["left", "right", "top"]].set_visible(False)

ax = axes[1]
cs_left = np.linspace(-1.5, -0.06, 240)
cs_right = np.linspace(0.06, 1.5, 240)
ax.plot(cs_left, [harmonic_fourth(a_h, b_h, c) for c in cs_left], color="#c92a2a", lw=2)
ax.plot(cs_right, [harmonic_fourth(a_h, b_h, c) for c in cs_right], color="#c92a2a", lw=2)
ax.axvline(0, color="#222", lw=1, ls="--")
ax.axhline(0, color="#888", lw=0.8)
ax.scatter([c_h], [d_h], color="#2b8a3e", s=58, zorder=5)
ax.text(c_h + 0.07, d_h, "chosen c", va="center", color="#2b8a3e")
ax.text(0.05, 5.5, "d crosses infinity", color="#c92a2a", ha="left")
ax.set_xlim(-1.55, 1.55)
ax.set_ylim(-6, 6)
ax.set_xlabel("third point c")
ax.set_ylabel("harmonic fourth point d")
ax.set_title("For a=-1, b=1, harmonic d=1/c")
ax.spines[["top", "right"]].set_visible(False)
harmonic_path = save_figure(fig, "harmonic-quadruple-infinity-crossing.png")
book_relative(harmonic_path), harmonic_cr, infinity_harmonic_cr


## Permuting the Four Entries

The cross-ratio is ordered. Swapping the inputs does not leave the value arbitrary; it moves the value through a six-element orbit generated by `x -> 1/x` and `x -> 1 - x`. This is the computational form of the chapter's permutation rules. The graph below is a proof scaffold: starting from one measured value, every other permutation value is forced by these two operations.


In [ ]:
lam_value = 2.4
orbit_nodes = {
    "lambda": lam_value,
    "1/lambda": 1 / lam_value,
    "1-lambda": 1 - lam_value,
    "1/(1-lambda)": 1 / (1 - lam_value),
    "lambda/(lambda-1)": lam_value / (lam_value - 1),
    "(lambda-1)/lambda": (lam_value - 1) / lam_value,
}
inv_edges = [("lambda", "1/lambda"), ("1-lambda", "1/(1-lambda)"), ("lambda/(lambda-1)", "(lambda-1)/lambda")]
one_minus_edges = [("lambda", "1-lambda"), ("1/lambda", "(lambda-1)/lambda"), ("1/(1-lambda)", "lambda/(lambda-1)")]
G = nx.Graph()
for node, value in orbit_nodes.items():
    G.add_node(node, value=value)
for u, v in inv_edges:
    G.add_edge(u, v, op="x -> 1/x")
for u, v in one_minus_edges:
    G.add_edge(u, v, op="x -> 1-x")

pos = nx.circular_layout(G)
fig, ax = plt.subplots(figsize=(7.4, 6.2))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#777", width=1.4)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#edf2ff", edgecolors="#4263eb", node_size=1850, linewidths=1.2)
labels = {node: f"{node}\n{value:.3f}" for node, value in orbit_nodes.items()}
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=8)
edge_labels = nx.get_edge_attributes(G, "op")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax, font_size=7, rotate=False, bbox=dict(fc="white", ec="none", alpha=0.75))
ax.set_title("Six cross-ratio values forced by permutation rules")
ax.axis("off")
permutation_path = save_figure(fig, "cross-ratio-permutation-orbit.png")
book_relative(permutation_path)


In [ ]:
alpha, beta, gamma, delta = sp.symbols("alpha beta gamma delta")
x1, y1, x2, y2 = sp.symbols("x1 y1 x2 y2")
a, b, c, d = sp.symbols("a b c d")

M_sym = sp.Matrix([[alpha, beta], [gamma, delta]])
u = sp.Matrix([x1, y1])
v = sp.Matrix([x2, y2])
bracket = lambda p, q: sp.Matrix.hstack(p, q).det()
det_factor_identity = sp.factor(bracket(M_sym * u, M_sym * v) - M_sym.det() * bracket(u, v))

cr_expr = ((a - c) * (b - d)) / ((a - d) * (b - c))
harmonic_solution = sp.solve(sp.Eq(cr_expr, -1), d)[0]
harmonic_identity = sp.factor(cr_expr.subs(d, harmonic_solution) + 1)
permutation_identity = sp.factor((a - b) * (c - d) - (a - c) * (b - d) + (a - d) * (b - c))

symbolic_checks = {
    "cross_ratio_definition": "[a,c][b,d] / ([a,d][b,c]) with [u,v] a 2 x 2 determinant",
    "determinant_factor_identity": "[M u, M v] - det(M)[u,v] simplifies to 0",
    "determinant_factor_identity_zero": bool(det_factor_identity == 0),
    "harmonic_fourth_formula": str(harmonic_solution),
    "harmonic_identity": "substituting the solved fourth point into (a,b;c,d)+1 simplifies to 0",
    "harmonic_identity_zero": bool(harmonic_identity == 0),
    "permutation_identity": "(a-b)(c-d) - (a-c)(b-d) + (a-d)(b-c) simplifies to 0",
    "permutation_identity_zero": bool(permutation_identity == 0),
}
symbolic_path = save_json(symbolic_checks, ARTIFACT_ROOT, "checks", "symbolic-cross-ratio-checks.json")
symbolic_checks


## Artifact Gallery

The generated artifacts are linked directly here so the notebook remains readable even before the code cells are re-executed.

![Projective line chart transition](../../artifacts/chapter-04-lines-and-cross-ratios/figures/projective-line-rp1-chart-transition.png)

![Mobius reparametrization preserves cross-ratio](../../artifacts/chapter-04-lines-and-cross-ratios/figures/mobius-reparametrization-cross-ratio.png)

![Harmonic quadruple and infinity crossing](../../artifacts/chapter-04-lines-and-cross-ratios/figures/harmonic-quadruple-infinity-crossing.png)

![Cross-ratio permutation orbit](../../artifacts/chapter-04-lines-and-cross-ratios/figures/cross-ratio-permutation-orbit.png)

Open the [interactive Mobius/infinity lab](../../artifacts/chapter-04-lines-and-cross-ratios/html/rp1-mobius-infinity-lab.html) and inspect the [cross-ratio invariance table](../../artifacts/chapter-04-lines-and-cross-ratios/tables/cross-ratio-invariance-table.csv) when reading outside an executed kernel.


In [ ]:
artifact_paths = [chart_transition_path, mobius_path, harmonic_path, permutation_path, html_path, table_path, symbolic_path]
for artifact in artifact_paths[:4]:
    display_artifact(artifact, width=760)
display_artifact(html_path, width="100%", height=520)
pd.read_csv(table_path)


## Applied Lab: Change the Reference Points

The small function below is the chapter in miniature. Give it four finite chart values and a projective `2 x 2` matrix. It measures the cross-ratio before and after the matrix action, reports where the matrix sends a finite point to infinity, and returns the residual. Try replacing `lab_matrix` with a matrix whose lower row makes one of the four denominators vanish; the homogeneous determinant computation still has a value as long as the ordered quadruple remains valid in `RP^1`.


In [ ]:
def reparametrization_lab(values, matrix):
    values = [float(v) for v in values]
    matrix = np.asarray(matrix, dtype=float)
    before = cross_ratio_hom(*(rp1(v) for v in values))
    image_vectors = [matrix @ rp1(v) for v in values]
    images = [affine_chart(v) for v in image_vectors]
    after = cross_ratio_hom(*image_vectors)
    pole = None
    if abs(matrix[1, 0]) > TOL:
        pole = float(-matrix[1, 1] / matrix[1, 0])
    return {
        "input_values": values,
        "image_values": images,
        "matrix": matrix.tolist(),
        "finite_pole": pole,
        "cross_ratio_before": float(before),
        "cross_ratio_after": float(after),
        "residual": float(abs(before - after)),
    }


lab_matrix = np.array([[0.7, -0.5], [0.4, 1.1]])
lab_report = reparametrization_lab([-1.2, -0.1, 0.65, 2.1], lab_matrix)
assert lab_report["residual"] < 1e-10
lab_report


In [ ]:
storyboard = {
    "chapter goal": "Build projective line coordinates and identify the cross-ratio as the invariant of projective reparametrization.",
    "source span read": "Perspectives on Projective Geometry, Chapter 4, Sections 4.1-4.4, printed pp. 67-78 / PDF pp. 89-100.",
    "concept inventory": [
        "line through two projective points as the span [lambda p + mu q]",
        "RP^1 homogeneous coordinates [x:y] and the finite/infinite chart split",
        "chart transition u=1/t on the overlap",
        "2 x 2 projective transformations and finite Mobius formulas",
        "cross-ratio as a determinant quotient independent of representative scaling",
        "cross-ratio invariance under projective matrices and perspectivities",
        "six cross-ratio values generated by input permutations",
        "harmonic quadruples with cross-ratio -1, including the infinite fourth point",
        "RP^2 determinant formula for collinear point quadruples",
    ],
    "library routing table": [
        {"concept": "RP^1 chart transition", "representation": "labeled ray and reciprocal-chart diagram", "library": "matplotlib", "why": "static 2D rays and chart labels need durable inspection", "fallback": "plain SVG or matplotlib-only line diagram"},
        {"concept": "Mobius reparametrization", "representation": "before/after axis map plus interactive graph", "library": "matplotlib and plotly", "why": "static artifact records the invariant; Plotly lets learners zoom near denominator poles", "fallback": "matplotlib pole plot with sampled matrices"},
        {"concept": "cross-ratio identities", "representation": "exact determinant and algebra checks", "library": "sympy", "why": "the chapter's proof moves are determinant cancellations", "fallback": "small numeric determinant tests"},
        {"concept": "permutation orbit", "representation": "six-node proof-state graph", "library": "networkx with matplotlib", "why": "the graph exposes the generators x -> 1/x and x -> 1-x", "fallback": "table of six formulas"},
    ],
    "visual sequence": [
        {"concept": "projective line coordinates and RP1 charts", "artifact_filename": "figures/projective-line-rp1-chart-transition.png", "learner_inspection_target": "Compare the finite chart t with the reciprocal chart u and locate [1:0].", "validation_invariant": "finite overlap satisfies t*u=1"},
        {"concept": "Mobius/projective reparametrization", "artifact_filename": "figures/mobius-reparametrization-cross-ratio.png", "learner_inspection_target": "Follow four marked points from input coordinates to image coordinates.", "validation_invariant": "cross-ratio before and after the matrix action agree"},
        {"concept": "infinity crossing under projective maps", "artifact_filename": "html/rp1-mobius-infinity-lab.html", "learner_inspection_target": "Zoom near a denominator pole and observe the vertical blow-up as an infinity crossing.", "validation_invariant": "tracked points retain the same cross-ratio for each matrix"},
        {"concept": "harmonic quadruples", "artifact_filename": "figures/harmonic-quadruple-infinity-crossing.png", "learner_inspection_target": "Watch the harmonic fourth point d=1/c cross infinity when c=0.", "validation_invariant": "finite and infinite harmonic examples both have cross-ratio -1"},
        {"concept": "cross-ratio permutation values", "artifact_filename": "figures/cross-ratio-permutation-orbit.png", "learner_inspection_target": "Use the two generator labels to recover all six ordered values.", "validation_invariant": "symbolic permutation identity reduces to zero"},
        {"concept": "numeric invariance summary", "artifact_filename": "tables/cross-ratio-invariance-table.csv", "learner_inspection_target": "Compare residuals for scaling, Mobius action, RP2 determinants, and projection.", "validation_invariant": "all residuals are below tolerance"},
    ],
    "artifact plan": [book_relative(path) for path in artifact_paths],
    "computational checks": [
        "representative scaling invariance",
        "2 x 2 determinant factor identity",
        "Mobius cross-ratio residual below 1e-10",
        "RP2 determinant formula residual below 1e-10",
        "perspective projection residual below 1e-10",
        "harmonic finite and infinity examples equal -1",
        "artifact existence, nonzero size, and nonblank raster statistics",
    ],
    "proof-visualization strategy": "Use determinant factorization as the algebraic proof scaffold and a six-node permutation graph as the proof-state view for ordered cross-ratio values.",
    "implementation notes": "All generated files stay under artifacts/chapter-04-lines-and-cross-ratios; no shared helpers, scripts, indexes, source page images, or textbook figures are edited or copied.",
    "risks": ["Plotly HTML is larger than static PNG because it is standalone", "finite chart labels can hide the projective point at infinity unless the reciprocal chart is shown"],
    "acceptance criteria": ["nbclient executes the notebook", "visual and final sanity JSON files contain only book-relative paths", "all numeric residuals are below tolerance", "the shared generic visual-renderer scaffold is absent"],
}
storyboard_path = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
book_relative(storyboard_path)


In [ ]:
static_artifacts = [chart_transition_path, mobius_path, harmonic_path, permutation_path]
visual_checks = {
    "chapter": 4,
    "all_files_exist": all(path.exists() for path in artifact_paths),
    "raster_artifacts": [relative_stats(path) for path in static_artifacts],
    "html_artifact": book_relative(html_path),
    "table_artifact": book_relative(table_path),
    "symbolic_artifact": book_relative(symbolic_path),
    "visual_count": len(artifact_paths),
    "chart_transition_reciprocal_error": chart_transition_error,
    "cross_ratio_error": max(mobius_error, scaled_error, rp2_error, projection_error, abs(harmonic_cr + 1), abs(infinity_harmonic_cr + 1)),
    "mobius_cross_ratio_before": float(cr_before),
    "mobius_cross_ratio_after": float(cr_after),
    "representative_scaling_residual": scaled_error,
    "rp2_determinant_residual": rp2_error,
    "projection_residual": projection_error,
    "harmonic_cross_ratio": float(harmonic_cr),
    "infinity_harmonic_cross_ratio": float(infinity_harmonic_cr),
    "symbolic_checks": symbolic_checks,
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")
visual_checks


## Takeaways

- A projective line coordinate is a chart choice, not the point itself. The same `[x:y]` can look finite in one chart and infinite in another.
- A `2 x 2` projective matrix changes line coordinates by a Mobius formula; its denominator identifies which finite chart value crosses infinity.
- The determinant definition of the cross-ratio is the right computational object because representative scaling and projective matrices only add factors that cancel.
- Harmonic quadruples are ordinary cross-ratio computations with value `-1`; the infinite point is a legitimate outcome, not a failed calculation.
- The order of the four points matters, but all permutation values are controlled by the two operations `x -> 1/x` and `x -> 1 - x`.


In [ ]:
assert_artifacts(artifact_paths, min_size=256)
assert visual_checks["all_files_exist"]
assert visual_checks["chart_transition_reciprocal_error"] < 1e-12
assert visual_checks["cross_ratio_error"] < 1e-10
assert visual_checks["representative_scaling_residual"] < 1e-10
assert visual_checks["rp2_determinant_residual"] < 1e-10
assert visual_checks["projection_residual"] < 1e-10
assert visual_checks["symbolic_checks"]["determinant_factor_identity_zero"]
assert visual_checks["symbolic_checks"]["harmonic_identity_zero"]
assert visual_checks["symbolic_checks"]["permutation_identity_zero"]
for item in visual_checks["raster_artifacts"]:
    assert item["file_size"] > 256
    assert item["width"] >= 200 and item["height"] >= 150
    assert item["pixel_std"] > 1.0

final_sanity = {
    "chapter": 4,
    "source_span": "printed pp. 67-78 / PDF pp. 89-100",
    "notebook": book_relative(NOTEBOOK_PATH),
    "artifacts": [book_relative(path) for path in artifact_paths],
    "checks": {
        "max_cross_ratio_residual": visual_checks["cross_ratio_error"],
        "chart_transition_reciprocal_error": visual_checks["chart_transition_reciprocal_error"],
        "symbolic_checks": visual_checks["symbolic_checks"],
        "all_artifacts_nonzero": True,
    },
    "notebook_executed": True,
}
final_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity
